    # Preprocessing DisasterWatch

    Membersihkan tipe data, tanggal, nilai kosong, duplikasi, dan menyimpan dataset bersih.
    **Nilai kosong tidak otomatis dianggap nol** kecuali pada kolom dampak yang dipilih.
    

In [ ]:
    from pathlib import Path
    import pandas as pd
    import numpy as np
    
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    INPUT = sorted((ROOT / "data" / "processed").glob("bnpb_bencana_*.csv"))[-1]
    OUT = ROOT / "data" / "processed"
    df = pd.read_csv(INPUT, encoding="utf-8-sig")
    print("Sebelum:", df.shape)
    

## 1. Standarisasi tipe data

In [ ]:
    numeric_cols = ["id_prov", "id_kab", "id_jenis_bencana", "tahun", "bulan", "tanggal",
                    "meninggal", "terluka", "hilang", "menderita", "mengungsi", "jumlah_kejadian",
                    "terendam", "rumah_rusak", "lahan", "longitude", "latitude"]
    for col in numeric_cols:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["date"] = pd.to_datetime(dict(year=df.tahun, month=df.bulan, day=df.tanggal), errors="coerce")
    for col in ["provinsi", "kabkot", "kejadian", "lokasi"]:
        df[col] = df[col].astype("string").str.strip()
    df["kejadian"] = df["kejadian"].str.upper()
    

## 2. Penanganan missing value dan duplikasi

In [ ]:
    # Kolom dampak yang kosong diperlakukan sebagai 0 hanya untuk kebutuhan agregasi/model.
    impact_cols = ["meninggal", "terluka", "hilang", "menderita", "mengungsi", "terendam",
                   "rumah_rusak", "lahan", "jumlah_kejadian"]
    for col in impact_cols:
        df[col] = df[col].fillna(0)
    df["has_coordinates"] = df[["longitude", "latitude"]].notna().all(axis=1)
    df = df.dropna(subset=["date", "kejadian", "provinsi", "kabkot"])
    before = len(df)
    df = df.drop_duplicates()
    print("Baris duplikat penuh dihapus:", before - len(df))
    print("Sesudah:", df.shape)
    

## 3. Simpan dataset bersih

In [ ]:
    output = OUT / "bnpb_disaster_clean.csv"
    df.to_csv(output, index=False, encoding="utf-8-sig")
    print(output)
    display(df.head())
    

### Catatan
Untuk analisis dampak, bedakan antara nilai 0 (tidak ada dampak) dan nilai kosong (belum dilaporkan). Jika diperlukan, buat versi kedua yang mempertahankan `NaN`.